In [30]:
import os
os.environ['NUMEXPR_MAX_THREADS'] = '35'
from functionsgpu_old import *

import numpy as np
import pandas as pd
import matplotlib
import seaborn as sns
import pickle
from plotting_beta import *
from plotting_betas import *
torch.set_default_device('cuda:1')
device = torch.device('cuda:1')

import warnings
warnings.filterwarnings("ignore")
tslen = 200

## Loading Raw Data 

In [72]:
def loading():
    with open('unaligned_betas.pkl','rb') as f:
        betas = pickle.load(f)
    with open('euclidean_mean.pkl','rb') as f:
        mu = pickle.load(f)
    return betas, mu

betas_raw, mu_all_t = loading()
betas_raw = betas_raw/1000
mu_all_t = mu_all_t/1000
print(betas_raw.shape, mu_all_t.shape)

(155, 32, 3, 200) (32, 3, 200)


In [28]:
## plotting one skeleton
%matplotlib qt5
x1 = torch.from_numpy(betas_raw[0]).to(device=device, dtype=dtype)
plot_one_traj(x1, zoom=1.5)

In [29]:
%matplotlib qt5
x2 = torch.from_numpy(betas_raw[10]).to(device=device, dtype=dtype)
plot_one_traj(x2, zoom=1.5)

In [73]:
mu_all_t = mu_all_t[np.newaxis, ...]
print(mu_all_t.shape)

(1, 32, 3, 200)


In [74]:
stroke_betas = betas_raw[:3]
healthy_betas = betas_raw[45:48]

raw_betas = np.concatenate((stroke_betas, healthy_betas, mu_all_t), axis=0)
print(raw_betas.shape)
beta_list = list(raw_betas)
print(len(beta_list), beta_list[0].shape)

(7, 32, 3, 200)
7 (32, 3, 200)


In [75]:
%matplotlib qt5
colors=[(1, 0, 0)] * 3 + [(0, 0.5, 1)] * 3 + [(0, 0, 0)]
p = plotting_betas_landmark(beta_list, colors, zoom=1.5)

In [76]:
p.to_image('result_figures/raw_skeletons.png')
p.to_image('result_figures/raw_skeletons.eps')

## Aligned Data

In [77]:
def loading(filename, tslen):
    with open('{}/betas_aligned{}.pkl'.format(filename, tslen), 'rb') as f:
        betas_aligned = pickle.load(f)
    with open('{}/mu{}.pkl'.format(filename, tslen), 'rb') as f:
        mu = pickle.load(f)
    with open('{}/tangent_vecs{}.pkl'.format(filename, tslen), 'rb') as f:
        tangent_vec_all = pickle.load(f)
    with open('{}/gammas{}.pkl'.format(filename, tslen), 'rb') as f:
        gammas_all = pickle.load(f)
    return betas_aligned, mu, tangent_vec_all, gammas_all

betas_aligned_all, mu_all_t, tangent_vec_all, gammas_all = loading('aligned_data',tslen)
mu_all_t_tensor = torch.from_numpy(mu_all_t).to(device=device, dtype=torch.float32)
print(len(betas_aligned_all), tangent_vec_all.shape, mu_all_t.shape)

155 (32, 3, 200, 155) (32, 3, 200)


In [78]:
# Wrap mu in a list - keep shape (32, 3, 200) to match betas_aligned_all elements
mu_all_t = [mu_all_t]
print(len(mu_all_t), mu_all_t[0].shape)

1 (32, 3, 200)


In [79]:
stroke_betas = betas_aligned_all[:3]
healthy_betas = betas_aligned_all[45:48]
abetas = stroke_betas+healthy_betas+mu_all_t
print(len(abetas), abetas[0].shape)

7 (32, 3, 200)


In [81]:
%matplotlib qt5
colors=[(1, 0, 0)] * 3 + [(0, 0.5, 1)] * 3 + [(0, 0, 0)]
p = plotting_betas_landmark(abetas, colors, zoom=5)

In [83]:
p.to_image('result_figures/a_skeletons.png')
p.to_image('result_figures/a_skeletons.eps')

## Mean Calculation

In [103]:
stroke_mean = np.mean(betas_aligned_all[0:44], axis=0)
healthy_mean = np.mean(betas_aligned_all[44:], axis=0)

print(stroke_mean.shape, healthy_mean.shape)

healthy_mean[:,1,:] = healthy_mean[:,1,:]+0.2

(32, 3, 200) (32, 3, 200)


In [108]:
%matplotlib qt5
colors=[(1, 0, 0), (0, 0.5, 1)]
betas_mean = [stroke_mean, healthy_mean]
p = plotting_betas_landmark(betas_mean, colors, zoom=5)

In [118]:
p.to_image('result_figures/means/frame8.png')
p.to_image('result_figures/means/frame8.eps')